In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [2]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 6 teams with confirmed lineups


### Load Model

In [3]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')

model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 4.5


### Load Player Data and Bookmaker Data

In [4]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Alex Sarr,Over,17.5,-137,2025-11-26,2025-11-25T23:48:12Z
1,Underdog,player_points,Alex Sarr,Under,17.5,-137,2025-11-26,2025-11-25T23:48:12Z
2,Underdog,player_points,Kristaps Porzingis,Over,19.5,-137,2025-11-26,2025-11-25T23:48:12Z
3,Underdog,player_points,Kristaps Porzingis,Under,19.5,-137,2025-11-26,2025-11-25T23:48:12Z
4,Underdog,player_points,Jalen Johnson,Over,23.5,-137,2025-11-26,2025-11-25T23:48:12Z


## Top EVs for 2 leg bets

### Underdog picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 47 players...
Processing 40 players...
Generated 708 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
375,Tyrese Maxey,LeBron James,32.5,19.5,-139,-115,26.23,15.88,0.781,0.735,under,under,0,68.83,0.344,High,Med
682,Jalen Duren,Reed Sheppard,18.5,14.5,-137,-137,21.26,11.93,0.661,0.655,over,under,0,27.32,0.137,High,High
11,Alex Sarr,Anthony Black,17.5,11.5,-120,-145,20.33,14.09,0.652,0.650,over,over,0,24.47,0.122,High,High
483,Trendon Watford,Tobias Harris,10.5,11.5,-135,-137,8.48,14.15,0.637,0.646,under,over,0,21.07,0.105,Med,High
559,James Harden,Jordan Walsh,24.5,5.5,-114,-137,27.11,6.86,0.634,0.626,over,over,0,16.57,0.083,High,Low
610,Brook Lopez,Moses Moody,5.5,11.5,-130,-137,7.49,13.58,0.634,0.617,over,over,0,14.96,0.075,Med,High
114,Zaccharie Risacher,Austin Reaves,10.5,22.5,-130,-125,12.60,24.98,0.615,0.627,over,over,0,13.33,0.067,High,High
187,Nickeil Alexander-Walker,Ausar Thompson,17.5,10.5,-136,-137,19.61,12.33,0.611,0.609,over,over,0,9.38,0.047,High,High
63,Kristaps Porziņģis,Duncan Robinson,19.5,10.5,-125,-137,17.54,12.18,0.608,0.601,under,over,0,7.54,0.038,High,High
143,Luke Kennard,Kawhi Leonard,6.5,20.5,-124,-114,7.62,22.06,0.583,0.589,over,over,0,0.96,0.005,Med,High


### Prizepicks picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 45 players...
Processing 41 players...
Generated 697 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
416,Tyrese Maxey,LeBron James,32.5,20.5,-139,-145,26.23,15.88,under,under,0.781,0.789,0.6038,0.199,0.197,0.266,81.15,0.406,1,8.09,5.75,High,Med,"(10.4, 42.1)","(4.6, 27.2)",0.05,0,81.1
525,Jared McCain,Jake LaRavia,12.5,6.5,-120,-130,9.13,10.83,under,over,0.729,0.746,0.5326,0.183,0.180,0.231,59.78,0.299,0,5.52,6.55,Med,High,"(0.0, 20.0)","(0.0, 23.7)",0.05,0,59.8
189,Corey Kispert,Rui Hachimura,13.0,10.5,-137,-136,9.52,14.58,under,over,0.728,0.736,0.5250,0.149,0.160,0.198,57.51,0.288,0,5.74,6.45,Med,High,"(0.0, 20.8)","(1.9, 27.2)",0.05,0,57.5
12,Jalen Johnson,Tristan da Silva,19.5,12.0,-137,-137,23.37,14.88,over,over,0.704,0.658,0.4537,0.125,0.080,0.126,36.11,0.181,0,7.24,7.08,High,High,"(9.2, 37.6)","(1.0, 28.8)",0.05,0,36.1
113,Alex Sarr,Anthony Black,17.5,11.5,-120,-145,20.33,14.09,over,over,0.652,0.650,0.4149,0.106,0.058,0.098,24.47,0.122,0,7.27,6.74,High,High,"(6.1, 34.6)","(0.9, 27.3)",0.05,0,24.5
318,Zaccharie Risacher,Brook Lopez,10.5,5.5,-130,-130,12.60,7.49,over,over,0.615,0.634,0.3821,0.050,0.069,0.069,14.62,0.073,0,7.19,5.82,High,Med,"(0.0, 26.7)","(0.0, 18.9)",0.05,0,14.6
619,Goga Bitadze,James Harden,5.5,24.5,-108,-114,6.74,27.11,over,over,0.605,0.634,0.3761,0.086,0.101,0.105,12.83,0.064,0,4.63,7.64,Low,High,"(0.0, 15.8)","(12.1, 42.1)",0.05,0,12.8
604,Trendon Watford,Austin Reaves,10.0,22.5,-137,-125,8.48,24.98,under,over,0.604,0.627,0.3711,0.026,0.071,0.057,11.33,0.057,0,5.77,7.67,Med,High,"(0.0, 19.8)","(9.9, 40.0)",0.05,0,11.3
65,Kristaps Porziņģis,Marcus Smart,19.0,6.5,-137,-105,17.54,8.19,under,over,0.581,0.610,0.3473,0.003,0.098,0.057,4.19,0.021,0,7.14,6.05,High,High,"(3.6, 31.5)","(0.0, 20.0)",0.05,0,4.2
393,Justin Champagnie,Kawhi Leonard,6.5,20.5,-135,-114,5.61,22.06,under,over,0.576,0.589,0.3328,0.002,0.057,0.033,-0.16,0.000,0,4.61,6.89,Low,High,"(0.0, 14.7)","(8.6, 35.6)",0.05,0,-0.2


## 3 leg parlay

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogTrios = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 47 players...
Processing 40 players...
Generated 7302 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
5122,Tyrese Maxey,LeBron James,Jalen Duren,32.5,19.5,18.5,26.23,15.88,21.26,0.781,0.735,0.661,under,under,over,0,104.96,0.210,High,Med,High
335,Alex Sarr,Anthony Black,Reed Sheppard,17.5,11.5,14.5,20.33,14.09,11.93,0.652,0.650,0.655,over,over,under,0,49.80,0.100,High,High,High
6166,Trendon Watford,Brook Lopez,Tobias Harris,10.5,5.5,11.5,8.48,7.49,14.15,0.637,0.634,0.646,under,over,over,0,40.96,0.082,Med,Med,High
6737,James Harden,Jordan Walsh,Moses Moody,24.5,5.5,11.5,27.11,6.86,13.58,0.634,0.626,0.617,over,over,over,0,32.08,0.064,High,Low,High
1801,Zaccharie Risacher,Austin Reaves,Ausar Thompson,10.5,22.5,10.5,12.60,24.98,12.33,0.615,0.627,0.609,over,over,over,0,26.71,0.053,High,High,High


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 45 players...
Processing 41 players...
Generated 6291 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
2305,Corey Kispert,Tyrese Maxey,LeBron James,13.0,32.5,20.5,9.52,26.23,15.88,0.728,0.781,0.789,under,under,under,0,142.07,0.284,Med,High,Med
287,Jalen Johnson,Jared McCain,Jake LaRavia,19.5,12.5,6.5,23.37,9.13,10.83,0.704,0.729,0.746,over,under,over,0,106.47,0.213,High,Med,High
1622,Alex Sarr,Tristan da Silva,Rui Hachimura,17.5,12.0,10.5,20.33,14.88,14.58,0.652,0.658,0.736,over,over,over,0,70.52,0.141,High,High,High
3898,Zaccharie Risacher,Anthony Black,Brook Lopez,10.5,11.5,5.5,12.60,14.09,7.49,0.615,0.650,0.634,over,over,over,0,36.78,0.074,High,High,Med
840,Kristaps Porziņģis,Goga Bitadze,James Harden,19.0,5.5,24.5,17.54,6.74,27.11,0.581,0.605,0.634,under,over,over,0,20.45,0.041,High,Low,High


In [9]:
# df = playerScoring('Trey Murphy III', s26, current_date, teamStarPlayer, projectedStartingFive)
# playerContext('Trey Murphy III', s26, current_date, projectedStartingFive, mainStartingFive, teamStarPlayer)